# VeriDoc: Forensic Deepfake Detection & Dual-Stream Benchmark
### Official Google Colab GPU Training & Scientific Evaluation Notebook

This notebook provides the **push-button, reproducible workflow** for:
1. Validating and preparing the **SDFVD dataset** (106 videos: 53 real, 53 fake).
2. Applying **face detection** (OpenCV Haar cascade) and resizing crops to $256 \times 256$ RGB.
3. Enforcing **strict pair-grouped splitting** (`v_i` and `vs_i` kept in the same split) to guarantee **zero identity/actor leakage** across splits.
4. Training the **MesoInception-4** baseline model with GPU acceleration (`BCEWithLogitsLoss`, AdamW, early stopping).
5. Training the proposed **DualStreamForensicNet** (combining spatial RGB and 2D Fourier FFT magnitude spectrum).
6. Conducting **rigorous scientific evaluation** on the held-out test split (crop-level & video-level aggregation, ROC-AUC, EER, ECE).
7. Generating head-to-head model comparison tables and saving weights directly to Google Drive.


## Step 1: Hardware & GPU Environment Verification


In [ ]:
# Check GPU availability and hardware specifications
!nvidia-smi

import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
else:
    print('WARNING: GPU not detected. Using CPU.')
    device = torch.device('cpu')


## Step 2: Install Required Dependencies


In [ ]:
# Install required packages
!pip install -q pydantic-settings opencv-python-headless scikit-learn matplotlib tabulate


## Step 3: Mount Google Drive & Locate Dataset Archive


In [ ]:
import os
import shutil
from pathlib import Path

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_mounted = True
except Exception as e:
    print(f'Drive mount skipped or running outside Colab: {e}')
    drive_mounted = False

# Search for SDFVD.zip in Google Drive or local content
search_locations = [
    Path('/content/drive/MyDrive/SDFVD.zip'),
    Path('/content/drive/MyDrive/VeriDoc/SDFVD.zip'),
    Path('/content/SDFVD.zip'),
]

zip_source = None
for p in search_locations:
    if p.exists() and p.is_file():
        zip_source = p
        print(f'Found SDFVD archive at: {zip_source} ({zip_source.stat().st_size:,} bytes)')
        break

local_zip = Path('/content/SDFVD.zip')
if zip_source and zip_source != local_zip:
    print(f'Copying {zip_source} to {local_zip} for high-speed local NVMe access...')
    shutil.copy2(zip_source, local_zip)

if not local_zip.exists():
    raise FileNotFoundError(
        'SDFVD.zip not found! Please upload SDFVD.zip directly to /content/SDFVD.zip '
        'or place it in your Google Drive at /content/drive/MyDrive/SDFVD.zip'
    )
print(f'Ready to process: {local_zip} ({local_zip.stat().st_size:,} bytes)')


## Step 4: Leakage-Safe Dataset Preparation (Face Crop & Split Generation)


In [ ]:
import zipfile
import re
import cv2
import json
import random
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

WORK_DIR = Path('/content/data_sdfvd')
RAW_DIR = WORK_DIR / 'raw_videos'
CROPS_DIR = WORK_DIR / 'crops'
RAW_DIR.mkdir(parents=True, exist_ok=True)
CROPS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Unpack videos safely
print('[PREP] Unpacking SDFVD.zip...')
with zipfile.ZipFile(local_zip, 'r') as zf:
    for member in zf.namelist():
        if member.endswith('.mp4'):
            filename = Path(member).name
            target = RAW_DIR / filename
            if not target.exists():
                with zf.open(member) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)

real_vids = sorted(list(RAW_DIR.glob('v*.mp4')))
fake_vids = sorted(list(RAW_DIR.glob('vs*.mp4')))
# Filter out vs from real list
real_vids = [v for v in real_vids if not v.name.startswith('vs')]

print(f'[PREP] Extracted: {len(real_vids)} Real videos, {len(fake_vids)} Fake videos.')

# 2. Strict Pair Grouping to Guarantee Zero Leakage
pairs = {}
for v in real_vids:
    idx = re.search(r'v(\d+)', v.name).group(1)
    pairs.setdefault(idx, {})['real'] = v
for v in fake_vids:
    idx = re.search(r'vs(\d+)', v.name).group(1)
    pairs.setdefault(idx, {})['fake'] = v

pair_keys = sorted(list(pairs.keys()), key=lambda x: int(x))
random.shuffle(pair_keys)

n_total = len(pair_keys)
n_train = int(round(n_total * 0.70))
n_val = int(round(n_total * 0.15))
n_test = n_total - n_train - n_val

train_keys = set(pair_keys[:n_train])
val_keys = set(pair_keys[n_train:n_train + n_val])
test_keys = set(pair_keys[n_train + n_val:])

video_splits = []
for k in pair_keys:
    split = 'train' if k in train_keys else ('val' if k in val_keys else 'test')
    if 'real' in pairs[k]:
        video_splits.append({'video_id': pairs[k]['real'].stem, 'subject_id': f'pair_{k}', 'label': 'REAL', 'split': split, 'path': str(pairs[k]['real'])})
    if 'fake' in pairs[k]:
        video_splits.append({'video_id': pairs[k]['fake'].stem, 'subject_id': f'pair_{k}', 'label': 'FAKE', 'split': split, 'path': str(pairs[k]['fake'])})

splits_df = pd.DataFrame(video_splits)
splits_df.to_csv(WORK_DIR / 'SDFVD_video_splits.csv', index=False)
print(f'[PREP] Video splits created: Train={len(train_keys)} pairs, Val={len(val_keys)} pairs, Test={len(test_keys)} pairs.')

# 3. Frame Sampling and Face Crop Extraction (256x256 RGB)
cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(cascade_path)

manifest_records = []
excluded_records = []

for idx, row in splits_df.iterrows():
    vpath = row['path']
    vid = row['video_id']
    sub = row['subject_id']
    lbl = row['label']
    splt = row['split']
    
    cap = cv2.VideoCapture(vpath)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        excluded_records.append({'video_id': vid, 'reason': 'unreadable_or_empty'})
        cap.release()
        continue
    
    sample_indices = np.linspace(0, total_frames - 1, 5, dtype=int)
    for frame_idx in sample_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret or frame is None:
            excluded_records.append({'video_id': vid, 'frame_idx': int(frame_idx), 'reason': 'frame_decode_failure'})
            continue
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(60, 60))
        if len(faces) == 0:
            excluded_records.append({'video_id': vid, 'frame_idx': int(frame_idx), 'reason': 'no_face_detected'})
            continue
        
        # Take largest face
        x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
        # Add 15% margin
        margin = int(0.15 * max(w, h))
        H, W = frame.shape[:2]
        x1 = max(0, x - margin)
        y1 = max(0, y - margin)
        x2 = min(W, x + w + margin)
        y2 = min(H, y + h + margin)
        
        face_crop = frame[y1:y2, x1:x2]
        face_crop = cv2.resize(face_crop, (256, 256), interpolation=cv2.INTER_AREA)
        
        crop_name = f'{vid}_frame{frame_idx:04d}_face0.jpg'
        crop_path = CROPS_DIR / crop_name
        cv2.imwrite(str(crop_path), face_crop)
        
        manifest_records.append({
            'path': str(crop_path),
            'label': lbl,
            'video_id': vid,
            'subject_id': sub,
            'split': splt
        })
    cap.release()

manifest_df = pd.DataFrame(manifest_records)
manifest_path = WORK_DIR / 'SDFVD_clean_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)

print(f'[PREP] Generated {len(manifest_df)} face crops across {len(splits_df)} videos.')
print(f'[PREP] Manifest saved to: {manifest_path}')
print(f'[PREP] Split counts: {manifest_df["split"].value_counts().to_dict()}')
print(f'[PREP] Excluded frames: {len(excluded_records)}')


## Step 5: Deep Learning Model Architectures


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. MesoInception-4 (Baseline)
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_1x1, out_3x3, out_5x5, out_7x7):
        super(InceptionBlock, self).__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1),
            nn.ReLU(inplace=True)
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_3x3, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_3x3, out_3x3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_5x5, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_5x5, out_5x5, kernel_size=3, padding=2, dilation=2),
            nn.ReLU(inplace=True)
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_7x7, kernel_size=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)

class MesoInception4(nn.Module):
    def __init__(self, num_classes=1):
        super(MesoInception4, self).__init__()
        self.inception1 = InceptionBlock(3, 1, 4, 4, 2)
        self.bn1 = nn.BatchNorm2d(11)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.inception2 = InceptionBlock(11, 2, 4, 4, 2)
        self.bn2 = nn.BatchNorm2d(12)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv3 = nn.Conv2d(12, 16, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm2d(16)
        self.relu3 = nn.ReLU(inplace=True)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv4 = nn.Conv2d(16, 16, kernel_size=5, padding=2)
        self.bn4 = nn.BatchNorm2d(16)
        self.relu4 = nn.ReLU(inplace=True)
        self.pool4 = nn.MaxPool2d(kernel_size=4, stride=4)
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(16 * 8 * 8, 16),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            nn.Linear(16, num_classes)
        )
        self.sigmoid = nn.Sigmoid()

    def forward_features(self, x):
        x = self.inception1(x)
        x = self.bn1(x)
        x = self.pool1(x)
        x = self.inception2(x)
        x = self.bn2(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.pool3(x)
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu4(x)
        x = self.pool4(x)
        return x

    def forward(self, x, return_features=False):
        feats = self.forward_features(x)
        logits = self.fc(feats)
        prob = self.sigmoid(logits)
        if return_features:
            return prob, feats
        return prob


# 2. DualStreamForensicNet (Spatial + 2D FFT Frequency Stream)
def compute_fft2d_spectrum_batch(x):
    gray = 0.2989 * x[:, 0:1, :, :] + 0.5870 * x[:, 1:2, :, :] + 0.1140 * x[:, 2:3, :, :]
    fft = torch.fft.fft2(gray)
    fft_shift = torch.fft.fftshift(fft)
    mag = torch.abs(fft_shift) + 1e-8
    log_mag = torch.log(mag)
    B = log_mag.shape[0]
    min_v = log_mag.view(B, -1).min(dim=1)[0].view(B, 1, 1, 1)
    max_v = log_mag.view(B, -1).max(dim=1)[0].view(B, 1, 1, 1)
    norm = (log_mag - min_v) / (max_v - min_v + 1e-8)
    return norm

class DualStreamForensicNet(nn.Module):
    def __init__(self, num_classes=1):
        super(DualStreamForensicNet, self).__init__()
        self.spatial_stream = MesoInception4(num_classes=num_classes)
        
        self.freq_stream = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((8, 8)),
            nn.Flatten()
        )
        
        self.fusion_classifier = nn.Sequential(
            nn.Linear((16 * 8 * 8) + (64 * 8 * 8), 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        spatial_feats = self.spatial_stream.forward_features(x)
        spatial_flat = torch.flatten(spatial_feats, start_dim=1)
        
        freq_input = compute_fft2d_spectrum_batch(x)
        freq_flat = self.freq_stream(freq_input)
        
        combined = torch.cat([spatial_flat, freq_flat], dim=1)
        logits = self.fusion_classifier(combined)
        return self.sigmoid(logits)

print('Architectures compiled successfully.')
print(f'MesoInception-4 Parameters: {sum(p.numel() for p in MesoInception4().parameters()):,}')
print(f'DualStreamForensicNet Parameters: {sum(p.numel() for p in DualStreamForensicNet().parameters()):,}')


## Step 6: Dataset Loader & Evaluation Framework


In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

class ForensicDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]
        img = cv2.imread(row['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        # Standard MesoNet normalization: mean=0.5, std=0.5 -> [-1, 1]
        tensor = (tensor - 0.5) / 0.5
        label = 1.0 if row['label'] == 'REAL' else 0.0
        return tensor, torch.tensor([label], dtype=torch.float32), row['video_id']

train_records = manifest_df[manifest_df['split'] == 'train'].to_dict('records')
val_records = manifest_df[manifest_df['split'] == 'val'].to_dict('records')
test_records = manifest_df[manifest_df['split'] == 'test'].to_dict('records')

train_loader = DataLoader(ForensicDataset(train_records), batch_size=16, shuffle=True)
val_loader = DataLoader(ForensicDataset(val_records), batch_size=16, shuffle=False)
test_loader = DataLoader(ForensicDataset(test_records), batch_size=16, shuffle=False)

print(f'DataLoader ready -> Train: {len(train_records)} crops, Val: {len(val_records)} crops, Test: {len(test_records)} crops.')


## Step 7: Training Engine with GPU Acceleration & Early Stopping


In [ ]:
import copy

def train_model(model, train_loader, val_loader, epochs=15, lr=1e-4, device='cuda', save_path=None):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCELoss()
    
    best_auc = -1.0
    best_weights = None
    history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_acc': []}
    
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for x, y, _ in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(y)
        
        train_loss = running_loss / len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        y_true, y_pred = [], []
        with torch.no_grad():
            for x, y, _ in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                loss = criterion(pred, y)
                val_loss += loss.item() * len(y)
                y_true.extend(y.cpu().numpy().flatten())
                y_pred.extend(pred.cpu().numpy().flatten())
        
        val_loss = val_loss / len(val_loader.dataset)
        val_auc = roc_auc_score(y_true, y_pred) if len(set(y_true)) > 1 else 0.5
        val_acc = accuracy_score(y_true, [1 if p >= 0.5 else 0 for p in y_pred])
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch [{epoch:02d}/{epochs:02d}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val Acc: {val_acc:.4f}')
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_weights = copy.deepcopy(model.state_dict())
            if save_path:
                torch.save(best_weights, save_path)
                print(f'  --> Saved best weights to {save_path} (AUC: {best_auc:.4f})')
                
    if best_weights:
        model.load_state_dict(best_weights)
    return model, history

WEIGHTS_DIR = Path('/content/drive/MyDrive/VeriDoc_Weights') if drive_mounted else Path('/content/weights')
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Weights will be preserved at: {WEIGHTS_DIR}')


## Step 8: Train MesoInception-4 (Baseline)


In [ ]:
meso_model = MesoInception4()
meso_save = WEIGHTS_DIR / 'meso_inception_best.pth'

print('--- Training MesoInception-4 ---')
meso_model, meso_hist = train_model(
    meso_model, train_loader, val_loader,
    epochs=10, lr=1e-4, device=device, save_path=meso_save
)


## Step 9: Train DualStreamForensicNet


In [ ]:
dual_model = DualStreamForensicNet()
dual_save = WEIGHTS_DIR / 'dual_stream_best.pth'

print('--- Training DualStreamForensicNet ---')
dual_model, dual_hist = train_model(
    dual_model, train_loader, val_loader,
    epochs=10, lr=1e-4, device=device, save_path=dual_save
)


## Step 10: Scientific Evaluation & Multi-Level Aggregation


In [ ]:
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.metrics import roc_curve

def evaluate_scientific(model, loader, device='cuda'):
    model.eval()
    y_true = []
    y_scores = []
    video_map = {}
    
    with torch.no_grad():
        for x, y, vids in loader:
            x = x.to(device)
            preds = model(x).cpu().numpy().flatten()
            labels = y.numpy().flatten()
            for p, l, v in zip(preds, labels, vids):
                y_scores.append(float(p))
                y_true.append(float(l))
                video_map.setdefault(v, {'scores': [], 'label': float(l)})['scores'].append(float(p))
                
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    
    # Crop-level metrics
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    crop_auc = roc_auc_score(y_true, y_scores)
    
    # EER
    eer = brentq(lambda x : 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    
    # Youden Index optimal threshold
    youden_idx = np.argmax(tpr - fpr)
    opt_th = thresholds[youden_idx]
    
    preds_05 = (y_scores >= 0.5).astype(int)
    preds_opt = (y_scores >= opt_th).astype(int)
    
    crop_acc = accuracy_score(y_true, preds_05)
    crop_f1 = f1_score(y_true, preds_05)
    crop_prec = precision_score(y_true, preds_05, zero_division=0)
    crop_rec = recall_score(y_true, preds_05, zero_division=0)
    
    # Video-level aggregation
    vid_true = []
    vid_scores = []
    for v, data in video_map.items():
        vid_scores.append(float(np.mean(data['scores'])))
        vid_true.append(data['label'])
    vid_true = np.array(vid_true)
    vid_scores = np.array(vid_scores)
    vid_auc = roc_auc_score(vid_true, vid_scores)
    vid_acc = accuracy_score(vid_true, (vid_scores >= 0.5).astype(int))
    
    return {
        'crop_auc': crop_auc,
        'crop_eer': eer,
        'crop_acc': crop_acc,
        'crop_f1': crop_f1,
        'crop_prec': crop_prec,
        'crop_rec': crop_rec,
        'optimal_threshold': opt_th,
        'vid_auc': vid_auc,
        'vid_acc': vid_acc,
        'fpr': fpr,
        'tpr': tpr
    }

meso_eval = evaluate_scientific(meso_model, test_loader, device=device)
dual_eval = evaluate_scientific(dual_model, test_loader, device=device)

print('Evaluation complete.')


## Step 11: Head-to-Head Benchmark Comparison


In [ ]:
from tabulate import tabulate
import matplotlib.pyplot as plt

table_data = [
    ['Architecture', 'MesoInception-4 (Baseline)', 'DualStreamForensicNet', 'Delta'],
    ['Input Modalities', 'Spatial RGB', 'Spatial RGB + 2D FFT Spectrum', '+1 Frequency Stream'],
    ['Parameters', f'{sum(p.numel() for p in meso_model.parameters()):,}', f'{sum(p.numel() for p in dual_model.parameters()):,}', f'+{sum(p.numel() for p in dual_model.parameters()) - sum(p.numel() for p in meso_model.parameters()):,}'],
    ['Crop ROC-AUC', f'{meso_eval["crop_auc"]:.4f}', f'{dual_eval["crop_auc"]:.4f}', f'{dual_eval["crop_auc"] - meso_eval["crop_auc"]:+.4f}'],
    ['Crop EER', f'{meso_eval["crop_eer"]:.2%}', f'{dual_eval["crop_eer"]:.2%}', f'{dual_eval["crop_eer"] - meso_eval["crop_eer"]:+.2%}'],
    ['Crop Accuracy (th=0.5)', f'{meso_eval["crop_acc"]:.2%}', f'{dual_eval["crop_acc"]:.2%}', f'{dual_eval["crop_acc"] - meso_eval["crop_acc"]:+.2%}'],
    ['Crop F1-Score', f'{meso_eval["crop_f1"]:.4f}', f'{dual_eval["crop_f1"]:.4f}', f'{dual_eval["crop_f1"] - meso_eval["crop_f1"]:+.4f}'],
    ['Video ROC-AUC', f'{meso_eval["vid_auc"]:.4f}', f'{dual_eval["vid_auc"]:.4f}', f'{dual_eval["vid_auc"] - meso_eval["vid_auc"]:+.4f}'],
    ['Video Accuracy', f'{meso_eval["vid_acc"]:.2%}', f'{dual_eval["vid_acc"]:.2%}', f'{dual_eval["vid_acc"] - meso_eval["vid_acc"]:+.2%}'],
    ['Optimal Threshold', f'{meso_eval["optimal_threshold"]:.4f}', f'{dual_eval["optimal_threshold"]:.4f}', 'N/A']
]

print(tabulate(table_data, headers='firstrow', tablefmt='github'))

# Plot ROC Curves
plt.figure(figsize=(8, 6))
plt.plot(meso_eval['fpr'], meso_eval['tpr'], label=f'MesoInception-4 (AUC = {meso_eval["crop_auc"]:.3f})', color='royalblue', lw=2)
plt.plot(dual_eval['fpr'], dual_eval['tpr'], label=f'DualStreamForensicNet (AUC = {dual_eval["crop_auc"]:.3f})', color='crimson', lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Receiver Operating Characteristic (ROC) - SDFVD Benchmark')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()


## Step 12: Export Weights for Local VeriDoc Integration


In [ ]:
print(f'Checkpoints saved to: {WEIGHTS_DIR}')
print('Files available:')
for f in WEIGHTS_DIR.glob('*.*'):
    print(f'  - {f.name} ({f.stat().st_size:,} bytes)')

instructions = """
To deploy in your local VeriDoc installation:
1. Download meso_inception_best.pth or dual_stream_best.pth
2. Place in backend/training/weights/
3. Update backend/.env:
   DEEPFAKE_MODEL_PATH=training/weights/meso_inception_best.pth
   DEEPFAKE_MODEL_NAME=MesoInception-4
   DEEPFAKE_MODEL_VERSION=sdfvd_v1
"""
print(instructions)
